# Silver — reconciliation input quality

| Scenario | Rule | Expected |
|---|---|---|
| Valid ERPNext line | Required IDs, date and exactly one positive debit or credit | One validated GL row |
| Statement cleansing | ISO date, grouped decimal amount, trim reference | One typed statement row |
| Bad statement date | Invalid calendar dates are rejected | One exception |
| Duplicate statement | Same transaction ID and same normalized values | One duplicate and no extra accepted row |
| Mapping | Exactly one applicable bank account | Baseline accepted; missing and ambiguous recorded |
| Replay | Replace the dedicated run snapshot | Same records and counts |

Run after both Bronze notebooks. All Silver outputs are scoped to the demo run. The next cells declare typed Delta schemas and preserve Bronze references.

In [ ]:
import hashlib
import json
import os
import sys
from decimal import Decimal
from pathlib import Path
from urllib.parse import urlsplit
from uuid import uuid4

import boto3
import requests
from botocore.config import Config
from botocore.exceptions import ClientError
from dotenv import dotenv_values
from pyspark.sql import SparkSession, functions as F

# Support launching Jupyter from either the repository root or the notebook folder.
cwd = Path.cwd().resolve()
candidates = [parent if parent.name == "data-platform" else parent / "data-platform"
              for parent in (cwd, *cwd.parents)]
env_path = next((folder / ".env" for folder in candidates if (folder / ".env").is_file()), None)
if env_path is None:
    raise FileNotFoundError("Create data-platform/.env and launch from the repository.")
cfg = dotenv_values(env_path, interpolate=False)

def required(name):
    value = cfg.get(name)
    if not value:
        raise ValueError(f"Set {name} in data-platform/.env")
    return value

endpoint = cfg.get("MINIO_ENDPOINT") or "http://192.168.29.250:9000"
bucket = cfg.get("BRONZE_BUCKET") or "bronze"
os.environ["AWS_ACCESS_KEY_ID"] = required("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = required("AWS_SECRET_ACCESS_KEY")
os.environ["PYSPARK_PYTHON"] = sys.executable

# Use path-style addressing for both Spark and the raw-object client.
s3 = boto3.client(
    "s3", endpoint_url=endpoint, region_name="us-east-1",
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    config=Config(s3={"addressing_style": "path"}),
)
spark = (
    SparkSession.builder.master("local[2]").appName("bronze-erpnext-gl")
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.3.3,org.apache.hadoop:hadoop-aws:3.3.4")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.hadoop.fs.s3a.endpoint", endpoint)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", str(urlsplit(endpoint).scheme == "https").lower())
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)

In [ ]:
import csv
import io
import re
from datetime import date
from decimal import InvalidOperation

run = required("DEMO_RUN_ID")
if not re.fullmatch(r"[A-Za-z0-9_-]{1,80}", run):
    raise ValueError("Invalid DEMO_RUN_ID")
silver = required("SILVER_BUCKET")
root = f"s3a://{silver}/tables/silver-demo/{run}"
bronze_gl = f"s3a://{bucket}/tables/erpnext/gl_entry"

def bronze_bytes(key):
    with s3.get_object(Bucket=bucket, Key=key)["Body"] as body:
        return body.read()

manifest = json.loads(bronze_bytes(f"raw/silver-demo/{run}/manifest.json"))
mappings = json.loads(bronze_bytes(manifest["mappings.json"]))
statement_uri = f"s3a://{bucket}/{manifest['statement.csv']}"
mapping_uri = f"s3a://{bucket}/{manifest['mappings.json']}"
statement = list(csv.DictReader(io.StringIO(bronze_bytes(manifest["statement.csv"]).decode("utf-8"))))
gl_rows = (spark.read.format("delta").load(bronze_gl)
           .where((F.col("voucher_no") == required("ERPNEXT_VOUCHER_NO")) &
                  (F.col("company") == required("ERPNEXT_COMPANY")) &
                  (F.col("account") == required("ERPNEXT_BANK_ACCOUNT"))).take(2))
if len(gl_rows) != 1:
    raise RuntimeError("Expected one matching ERPNext Bronze record.")

def amount(value):
    value = str(value).strip()
    if not re.fullmatch(r"[0-9]{1,16}(?:\.[0-9]{1,2})?|[0-9]{1,3}(?:,[0-9]{3})+(?:\.[0-9]{1,2})?", value):
        raise ValueError("INVALID_AMOUNT")
    number = Decimal(value.replace(",", ""))
    if number >= Decimal(10)**16:
        raise ValueError("INVALID_AMOUNT")
    return number

def posting_date(value):
    try:
        return date.fromisoformat(value.strip())
    except (ValueError, AttributeError):
        raise ValueError("INVALID_POSTING_DATE") from None

def field(value):
    if not isinstance(value, str) or not value.strip():
        raise ValueError("MISSING_REQUIRED_FIELD")
    return value.strip()

errors, gl, clean, links = [], [], {}, []
duplicates = 0
for row in gl_rows:
    try:
        debit, credit = amount(row.debit), amount(row.credit)
        if (debit > 0) == (credit > 0):
            raise ValueError("INVALID_DEBIT_CREDIT")
        gl.append((field(row.source_instance), field(row.name), field(row.company),
                   field(row.account), posting_date(row.posting_date), debit, credit, field(row.raw_uri)))
    except (ValueError, InvalidOperation) as exc:
        errors.append(("erpnext", str(exc), bronze_gl, str(row.name)))

for number, row in enumerate(statement, 2):
    try:
        key = field(row.get("transaction_id"))
        record = (key, field(row.get("bank_account_id")), posting_date(row.get("posting_date")),
                  amount(row.get("amount")), (row.get("reference") or "").strip(), statement_uri)
        if key in clean:
            if clean[key] != record:
                raise ValueError("CONFLICTING_TRANSACTION_ID")
            duplicates += 1
        else:
            clean[key] = record
    except (ValueError, InvalidOperation) as exc:
        errors.append(("statement", str(exc), statement_uri, str(number)))

# Check the mappings against the actual GL account for each scenario.
for scenario in ("baseline", "missing", "ambiguous"):
    for _, gl_id, company, account, *_ in gl:
        candidates = {entry["bank_account_id"] for entry in mappings
                      if entry["scenario"] == scenario and entry["company"] == company
                      and entry["ledger_account"] == account}
        if len(candidates) == 1:
            links.append((scenario, gl_id, candidates.pop(), mapping_uri))
        else:
            reason = "MISSING_MAPPING" if not candidates else "AMBIGUOUS_MAPPING"
            errors.append(("mapping", reason, mapping_uri, f"{scenario}:{gl_id}"))

statement_rejected = sum(row[0] == "statement" for row in errors)
assert len(statement) == len(clean) + duplicates + statement_rejected
assert (len(gl), len(clean), duplicates, statement_rejected, len(links)) == (1, 1, 1, 1, 1)
assert {row[1] for row in errors} == {"INVALID_POSTING_DATE", "MISSING_MAPPING", "AMBIGUOUS_MAPPING"}
print({"statement_input": len(statement), "statement_accepted": len(clean),
       "statement_quarantined": statement_rejected, "statement_duplicates": duplicates})

In [ ]:
# Explicit schemas enforce dates, fixed-precision amounts, and lineage fields.
tables = {
    "erpnext_gl": (gl, "source_instance STRING, gl_id STRING, company STRING, ledger_account STRING, "
                   "posting_date DATE, debit DECIMAL(18,2), credit DECIMAL(18,2), raw_uri STRING"),
    "statement_transactions": (list(clean.values()),
        "transaction_id STRING, bank_account_id STRING, posting_date DATE, amount DECIMAL(18,2), "
        "reference STRING, raw_uri STRING"),
    "account_mapping_checks": (links, "scenario STRING, gl_id STRING, bank_account_id STRING, raw_uri STRING"),
    "exceptions": (errors, "source STRING, reason STRING, bronze_uri STRING, record_locator STRING"),
}
for name, (records, schema) in tables.items():
    uri = f"{root}/{name}"
    frame = spark.createDataFrame(records, schema)
    # One scenario has its own snapshot. Reruns replace only that snapshot.
    for _ in range(2):
        frame.write.format("delta").mode("overwrite").save(uri)
        actual = spark.read.format("delta").load(uri)
        if actual.schema.simpleString() != frame.schema.simpleString() or actual.count() != len(records):
            raise RuntimeError(f"Schema or replay verification failed for {name}")
print("PASS: Silver schema, quarantine, mapping scenarios and replay verified")